___

___

# Proyecto de clasificación de texto
Ahora estamos en el punto en el que deberíamos poder:
* Leer en una colección de documentos - un *corpus*
* Transforme texto en datos vectoriales numéricos usando una canalización
* Crear un clasificador
* Ajustar/entrenar al clasificador
* Probar el clasificador con nuevos datos
* Evaluar el desempeño

Para este proyecto, utilizaremos el conjunto de datos de polaridad de Cornell University Movie Review v2.0 obtenido de http://www.cs.cornell.edu/people/pabo/movie-review-data/

En este ejercicio, intentaremos desarrollar un modelo de clasificación como lo hicimos para el conjunto de datos SMSSpamCollection; es decir, intentaremos predecir las etiquetas Positivo/Negativo basándose únicamente en el contenido del texto. En una próxima sección, aplicaremos *Análisis de opinión* para entrenar modelos que tengan una comprensión más profunda de cada revisión.

## Realizar importaciones y cargar el conjunto de datos
El conjunto de datos contiene el texto de 2000 reseñas de películas. 1000 son positivos, 1000 son negativos y el texto se ha procesado previamente como un archivo delimitado por tabuladores.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv('../datos/moviereviews.tsv', sep='\t')
df.head()

,label,review
0,neg,how do films like mouse hunt get into theatres...
1,neg,some talented actresses are blessed with a dem...
2,pos,this has been an extraordinary year for austra...
3,pos,according to hollywood movies made in last few...
4,neg,my first press screening of 1998 and already i...


In [2]:
len(df)

2000

### Echemos un vistazo a una reseña. Este está etiquetado como "negativo":

In [3]:
from IPython.display import Markdown, display
display(Markdown('> '+df['review'][0]))

> how do films like mouse hunt get into theatres ? 
isn't there a law or something ? 
this diabolical load of claptrap from steven speilberg's dreamworks studio is hollywood family fare at its deadly worst . 
mouse hunt takes the bare threads of a plot and tries to prop it up with overacting and flat-out stupid slapstick that makes comedies like jingle all the way look decent by comparison . 
writer adam rifkin and director gore verbinski are the names chiefly responsible for this swill . 
the plot , for what its worth , concerns two brothers ( nathan lane and an appalling lee evens ) who inherit a poorly run string factory and a seemingly worthless house from their eccentric father . 
deciding to check out the long-abandoned house , they soon learn that it's worth a fortune and set about selling it in auction to the highest bidder . 
but battling them at every turn is a very smart mouse , happy with his run-down little abode and wanting it to stay that way . 
the story alternates between unfunny scenes of the brothers bickering over what to do with their inheritance and endless action sequences as the two take on their increasingly determined furry foe . 
whatever promise the film starts with soon deteriorates into boring dialogue , terrible overacting , and increasingly uninspired slapstick that becomes all sound and fury , signifying nothing . 
the script becomes so unspeakably bad that the best line poor lee evens can utter after another run in with the rodent is : " i hate that mouse " . 
oh cringe ! 
this is home alone all over again , and ten times worse . 
one touching scene early on is worth mentioning . 
we follow the mouse through a maze of walls and pipes until he arrives at his makeshift abode somewhere in a wall . 
he jumps into a tiny bed , pulls up a makeshift sheet and snuggles up to sleep , seemingly happy and just wanting to be left alone . 
it's a magical little moment in an otherwise soulless film . 
a message to speilberg : if you want dreamworks to be associated with some kind of artistic credibility , then either give all concerned in mouse hunt a swift kick up the arse or hire yourself some decent writers and directors . 
this kind of rubbish will just not do at all . 


## Comprobar si faltan valores:
Intencionalmente hemos incluido registros con datos faltantes. Algunos tienen valores NaN, otros tienen cadenas cortas compuestas solo de espacios. Esto podría suceder si un revisor se negó a proporcionar un comentario con su revisión. Mostraremos dos formas de usar pandas para identificar y eliminar registros que contienen datos vacíos.
* Los registros de NaN se manejan de manera eficiente con [.isnull()](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.isnull.html) y [.dropna()](https: //pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.dropna.html)
* Las cadenas que contienen solo espacios en blanco se pueden manejar con [.isspace()](https://docs.python.org/3/library/stdtypes.html#str.isspace), [.itertuples()](https:/ /pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.itertuples.html) y [.drop()](https://pandas.pydata.org/pandas-docs/stable/generated/ pandas.DataFrame.drop.html)

### Detectar y eliminar valores de NaN:

In [4]:
# Comprobar la existencia de valores NaN en una celda:
df.isnull().sum()

label      0
review    35
dtype: int64

35 registros muestran **NaN** (esto significa "no es un número" y es equivalente a *Ninguno*). Estos se eliminan fácilmente usando la función pandas `.dropna()`.
<div class="alert alert-info" style="margin: 20px">PRECAUCIÓN: Al configurar inplace=True, afectamos permanentemente el DataFrame actualmente en la memoria, y esto no se puede deshacer. Sin embargo, *no* afecta los datos de la fuente original. Si lo necesitáramos, siempre podríamos cargar el DataFrame original desde cero.</div>

In [5]:
df.dropna(inplace=True)

len(df)

1965

### Detectar y eliminar cadenas vacías
Técnicamente, estamos tratando con cadenas de "solo espacios en blanco". Si el archivo .tsv original hubiera contenido cadenas vacías, pandas **.read_csv()** habría asignado valores NaN a esas celdas de forma predeterminada.

Para detectar estas cadenas, necesitamos iterar sobre cada fila en el DataFrame. El método pandas **.itertuples()** es una buena herramienta para esto, ya que proporciona acceso a todos los campos. Para abreviar, asignaremos los nombres `i`, `lb` y `rv` a las columnas `index`, `label` y `review`.

In [6]:
blanks = []  # comenzar con una lista vacía

for i,lb,rv in df.itertuples():  # itera sobre el DataFrame
    if type(rv)==str:            # evita NaN 
        if rv.isspace():         # prueba si hay espacios en blanco para 'review'
            blanks.append(i)     # agrega números de índice coincidentes a la lista
        
print(len(blanks), 'blanks: ', blanks)

27 blanks:  [57, 71, 147, 151, 283, 307, 313, 323, 343, 351, 427, 501, 633, 675, 815, 851, 977, 1079, 1299, 1455, 1493, 1525, 1531, 1763, 1851, 1905, 1993]


A continuación, pasaremos nuestra lista de números de índice al método **.drop()** y estableceremos `inplace=True` para que el cambio sea permanente.

In [7]:
df.drop(blanks, inplace=True)

len(df)

1938

¡Excelente! Bajamos 62 registros de los 2000 originales. Sigamos con el análisis.

## Eche un vistazo rápido a la columna `label`:

In [8]:
df['label'].value_counts()

neg    969
pos    969
Name: label, dtype: int64

## Dividir los datos en conjuntos de entrenamiento y prueba:

In [9]:
from sklearn.model_selection import train_test_split

X = df['review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

## Cree pipelines para vectorizar los datos, luego entrene y ajuste un modelo
Ahora que tenemos conjuntos para entrenar y probar, desarrollaremos una selección de canalizaciones, cada una con un modelo diferente.

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Naïve Bayes:
text_clf_nb = Pipeline([('tfidf', TfidfVectorizer()),
                     ('clf', MultinomialNB()),
])

# Linear SVC:
text_clf_lsvc = Pipeline([('tfidf', TfidfVectorizer()),
                     ('clf', LinearSVC()),
])

## Alimente los datos de entrenamiento a través de la primera canalización
Primero ejecutaremos Naive Bayes

In [11]:
text_clf_nb.fit(X_train, y_train)

Pipeline(memory=None,
     steps=[('tfidf', TfidfVectorizer(analyzer='word', binary=False, decode_error='strict',
        dtype=<class 'numpy.float64'>, encoding='utf-8', input='content',
        lowercase=True, max_df=1.0, max_features=None, min_df=1,
        ngram_range=(1, 1), norm='l2', preprocessor=None, smooth_idf=True,...rue,
        vocabulary=None)), ('clf', MultinomialNB(alpha=1.0, class_prior=None, fit_prior=True))])

## Ejecutar predicciones y analizar los resultados (naive Bayes)

In [12]:
# Formar un conjunto de predicción
predictions = text_clf_nb.predict(X_test)

In [13]:
# Reportar la matriz de confusión
from sklearn import metrics
print(metrics.confusion_matrix(y_test,predictions))

[[287  21]
 [130 202]]


In [14]:
# Imprimir un informe de clasificación
print(metrics.classification_report(y_test,predictions))

              precision    recall  f1-score   support

         neg       0.69      0.93      0.79       308
         pos       0.91      0.61      0.73       332

   micro avg       0.76      0.76      0.76       640
   macro avg       0.80      0.77      0.76       640
weighted avg       0.80      0.76      0.76       640



In [15]:
# Imprime la precisión general
print(metrics.accuracy_score(y_test,predictions))

0.7640625


Naive Bayes nos dio resultados mejores que el promedio con un 76,4% para clasificar las reseñas como positivas o negativas basándose únicamente en el texto. Veamos si podemos hacerlo mejor.

## Alimente los datos de entrenamiento a través de la segunda canalización
A continuación, ejecutaremos Linear SVC

In [16]:
text_clf_lsvc.fit(X_train, y_train)

Pipeline(memory=None,
     steps=[('tfidf', TfidfVectorizer(analyzer='word', binary=False, decode_error='strict',
        dtype=<class 'numpy.float64'>, encoding='utf-8', input='content',
        lowercase=True, max_df=1.0, max_features=None, min_df=1,
        ngram_range=(1, 1), norm='l2', preprocessor=None, smooth_idf=True,...ax_iter=1000,
     multi_class='ovr', penalty='l2', random_state=None, tol=0.0001,
     verbose=0))])

## Ejecutar predicciones y analizar los resultados (Linear SVC)

In [17]:
# Formar un conjunto de predicción
predictions = text_clf_lsvc.predict(X_test)

In [18]:
# Reportar la matriz de confusión
from sklearn import metrics
print(metrics.confusion_matrix(y_test,predictions))

[[259  49]
 [ 49 283]]


In [19]:
# Imprimir un informe de clasificación
print(metrics.classification_report(y_test,predictions))

              precision    recall  f1-score   support

         neg       0.84      0.84      0.84       308
         pos       0.85      0.85      0.85       332

   micro avg       0.85      0.85      0.85       640
   macro avg       0.85      0.85      0.85       640
weighted avg       0.85      0.85      0.85       640



In [20]:
# Imprime la precisión general
print(metrics.accuracy_score(y_test,predictions))

0.846875


¡Nada mal! Basándonos únicamente en el texto, clasificamos correctamente las reseñas como positivas o negativas **84,7 %** de las veces. En una próxima sección, intentaremos mejorar aún más esta puntuación mediante la realización de *análisis de opinión* en las reseñas.

## Tema avanzado: agregar stop words a CountVectorizer
De forma predeterminada, **CountVectorizer** y **TfidfVectorizer** *no* filtran palabras vacías. Sin embargo, ofrecen algunas configuraciones opcionales, incluida la transmisión de su propia lista de stop words.
<div class="alert alert-info" style="margin: 20px">PRECAUCIÓN: Hay algunos [problemas conocidos](http://aclweb.org/anthology/W18-2502) que usan stop words integradas de Scikit-learn. Algunas palabras que se filtran pueden, de hecho, ayudar en la clasificación. En esta sección, pasaremos nuestra propia lista de stop words, para que sepamos exactamente qué se está filtrando.</div>

La clase [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) acepta los siguientes argumentos:
> *CountVectorizer(input='content', encoding='utf-8', decode_error='strict', strip_accents=Ninguno, minúsculas=Verdadero, preprocesador=Ninguno, tokenizador=Ninguno, **stop_words=Ninguno**, token_pattern= '(?u)\b\w\w+\b', ngram_range=(1, 1), analizador='palabra', max_df=1.0, min_df=1, max_features=Ninguno, vocabulario=Ninguno, binario=Falso, dtype =<clase 'numpy.int64'>)*

[TfidVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) admite los mismos argumentos y más. En *stop_words* tenemos las siguientes opciones:
> stop_words : *string {'english'}, list, o None (predeterminado)*

Es decir, podemos ejecutar `TfidVectorizer(stop_words='english')` para aceptar la lista integrada de scikit-learn,<br>
o `TfidVectorizer(stop_words=[a, and, the])` para filtrar estas tres palabras. En la práctica, asignaríamos nuestra lista a una variable y la pasaríamos en su lugar.

La lista integrada de Scikit-learn contiene 318 palabras vacías:
> <pre>from sklearn.feature_extraction import text
> print(text.ENGLISH_STOP_WORDS)</pre>
['a', 'about', 'above', 'across', 'after', 'afterwards', 'again', 'against', 'all', 'almost', 'alone', 'along', 'already', 'also', 'although', 'always', 'am', 'among', 'amongst', 'amoungst', 'amount', 'an', 'and', 'another', 'any', 'anyhow', 'anyone', 'anything', 'anyway', 'anywhere', 'are', 'around', 'as', 'at', 'back', 'be', 'became', 'because', 'become', 'becomes', 'becoming', 'been', 'before', 'beforehand', 'behind', 'being', 'below', 'beside', 'besides', 'between', 'beyond', 'bill', 'both', 'bottom', 'but', 'by', 'call', 'can', 'cannot', 'cant', 'co', 'con', 'could', 'couldnt', 'cry', 'de', 'describe', 'detail', 'do', 'done', 'down', 'due', 'during', 'each', 'eg', 'eight', 'either', 'eleven', 'else', 'elsewhere', 'empty', 'enough', 'etc', 'even', 'ever', 'every', 'everyone', 'everything', 'everywhere', 'except', 'few', 'fifteen', 'fifty', 'fill', 'find', 'fire', 'first', 'five', 'for', 'former', 'formerly', 'forty', 'found', 'four', 'from', 'front', 'full', 'further', 'get', 'give', 'go', 'had', 'has', 'hasnt', 'have', 'he', 'hence', 'her', 'here', 'hereafter', 'hereby', 'herein', 'hereupon', 'hers', 'herself', 'him', 'himself', 'his', 'how', 'however', 'hundred', 'i', 'ie', 'if', 'in', 'inc', 'indeed', 'interest', 'into', 'is', 'it', 'its', 'itself', 'keep', 'last', 'latter', 'latterly', 'least', 'less', 'ltd', 'made', 'many', 'may', 'me', 'meanwhile', 'might', 'mill', 'mine', 'more', 'moreover', 'most', 'mostly', 'move', 'much', 'must', 'my', 'myself', 'name', 'namely', 'neither', 'never', 'nevertheless', 'next', 'nine', 'no', 'nobody', 'none', 'noone', 'nor', 'not', 'nothing', 'now', 'nowhere', 'of', 'off', 'often', 'on', 'once', 'one', 'only', 'onto', 'or', 'other', 'others', 'otherwise', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 'part', 'per', 'perhaps', 'please', 'put', 'rather', 're', 'same', 'see', 'seem', 'seemed', 'seeming', 'seems', 'serious', 'several', 'she', 'should', 'show', 'side', 'since', 'sincere', 'six', 'sixty', 'so', 'some', 'somehow', 'someone', 'something', 'sometime', 'sometimes', 'somewhere', 'still', 'such', 'system', 'take', 'ten', 'than', 'that', 'the', 'their', 'them', 'themselves', 'then', 'thence', 'there', 'thereafter', 'thereby', 'therefore', 'therein', 'thereupon', 'these', 'they', 'thick', 'thin', 'third', 'this', 'those', 'though', 'three', 'through', 'throughout', 'thru', 'thus', 'to', 'together', 'too', 'top', 'toward', 'towards', 'twelve', 'twenty', 'two', 'un', 'under', 'until', 'up', 'upon', 'us', 'very', 'via', 'was', 'we', 'well', 'were', 'what', 'whatever', 'when', 'whence', 'whenever', 'where', 'whereafter', 'whereas', 'whereby', 'wherein', 'whereupon', 'wherever', 'whether', 'which', 'while', 'whither', 'who', 'whoever', 'whole', 'whom', 'whose', 'why', 'will', 'with', 'within', 'without', 'would', 'yet', 'you', 'your', 'yours', 'yourself', 'yourselves']

Sin embargo, hay palabras en esta lista que pueden influir en la clasificación de las reseñas de películas. Con esto en mente, reduzcamos la lista a solo 60 palabras:

In [21]:
stopwords = ['a', 'about', 'an', 'and', 'are', 'as', 'at', 'be', 'been', 'but', 'by', 'can', \
             'even', 'ever', 'for', 'from', 'get', 'had', 'has', 'have', 'he', 'her', 'hers', 'his', \
             'how', 'i', 'if', 'in', 'into', 'is', 'it', 'its', 'just', 'me', 'my', 'of', 'on', 'or', \
             'see', 'seen', 'she', 'so', 'than', 'that', 'the', 'their', 'there', 'they', 'this', \
             'to', 'was', 'we', 'were', 'what', 'when', 'which', 'who', 'will', 'with', 'you']

Now let's repeat the process above and see if the removal of stopwords improves or impairs our score.

In [ ]:
# NO NECESITA EJECUTAR ESTA CELDA A MENOS QUE HAYA
# ABIERTO RECIENTEMENTEMENTE ESTE NOTEBOOK O REINICIÓ EL KERNEL:

import numpy as np
import pandas as pd

df = pd.read_csv('../datos/moviereviews.tsv', sep='\t')
df.dropna(inplace=True)
blanks = []
for i,lb,rv in df.itertuples():
    if type(rv)==str:
        if rv.isspace():
            blanks.append(i)
df.drop(blanks, inplace=True)
from sklearn.model_selection import train_test_split
X = df['review']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn import metrics

In [22]:
# EJECUTE ESTA CELDA PARA AGREGAR PALABRAS IMPORTANTES AL PIPELINE:
text_clf_lsvc2 = Pipeline([('tfidf', TfidfVectorizer(stop_words=stopwords)),
                     ('clf', LinearSVC()),
])
text_clf_lsvc2.fit(X_train, y_train)

Pipeline(memory=None,
     steps=[('tfidf', TfidfVectorizer(analyzer='word', binary=False, decode_error='strict',
        dtype=<class 'numpy.float64'>, encoding='utf-8', input='content',
        lowercase=True, max_df=1.0, max_features=None, min_df=1,
        ngram_range=(1, 1), norm='l2', preprocessor=None, smooth_idf=True,...ax_iter=1000,
     multi_class='ovr', penalty='l2', random_state=None, tol=0.0001,
     verbose=0))])

In [23]:
predictions = text_clf_lsvc2.predict(X_test)
print(metrics.confusion_matrix(y_test,predictions))

[[256  52]
 [ 48 284]]


In [24]:
print(metrics.classification_report(y_test,predictions))

              precision    recall  f1-score   support

         neg       0.84      0.83      0.84       308
         pos       0.85      0.86      0.85       332

   micro avg       0.84      0.84      0.84       640
   macro avg       0.84      0.84      0.84       640
weighted avg       0.84      0.84      0.84       640



In [25]:
print(metrics.accuracy_score(y_test,predictions))

0.84375


Nuestra puntuación no cambió mucho. Pasamos del 84,7 % sin filtrar stop words al 84,4 % después de agregar un filtro de stop words a nuestra canalización. Tenga en cuenta que las reseñas de películas de 2000 son un conjunto de datos relativamente pequeño. La verdadera ventaja de eliminar stop words es mejorar la velocidad de procesamiento; dependiendo del tamaño del corpus, podría ahorrar horas.

## Introducir nuevos datos en un modelo entrenado
Una vez que hemos desarrollado un modelo bastante preciso, es hora de alimentar nuevos datos a través de él. En esta última sección, escribiremos nuestra propia revisión y veremos con qué precisión nuestro modelo le asigna una etiqueta "positiva" o "negativa".

### First, train the model

In [ ]:
# NO NECESITA EJECUTAR ESTA CELDA A MENOS QUE TENGA
# ABRIR RECIENTEMENTE ESTE NOTEBOOK O REINICIA EL KERNEL:

import numpy as np
import pandas as pd

df = pd.read_csv('../datos/moviereviews.tsv', sep='\t')
df.dropna(inplace=True)
blanks = []
for i,lb,rv in df.itertuples():
    if type(rv)==str:
        if rv.isspace():
            blanks.append(i)
df.drop(blanks, inplace=True)
from sklearn.model_selection import train_test_split
X = df['review']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn import metrics

# Naïve Bayes:
text_clf_nb = Pipeline([('tfidf', TfidfVectorizer()),
                     ('clf', MultinomialNB()),
])

# Linear SVC:
text_clf_lsvc = Pipeline([('tfidf', TfidfVectorizer()),
                     ('clf', LinearSVC()),
])

# Entrene ambos modelos en el conjunto de entrenamiento moviereviews.tsv:
text_clf_nb.fit(X_train, y_train)
text_clf_lsvc.fit(X_train, y_train)

### A continuación, introduzca nuevos datos en el método `predict()` del modelo

In [26]:
myreview = "A movie I really wanted to love was terrible. \
I'm sure the producers had the best intentions, but the execution was lacking."

In [ ]:
# O use este espacio para escribir su propia reseña. Experimente con diferentes longitudes y estilos de escritura.
# myreview = 


In [27]:
print(text_clf_nb.predict([myreview]))  # be sure to put "myreview" inside square brackets

['neg']


In [28]:
print(text_clf_lsvc.predict([myreview]))

['neg']


¡Excelente! Ahora debería poder crear canalizaciones de clasificación de texto en scikit-learn, aplicar una variedad de algoritmos como naive Bayes y Linear SVC, manejar stop words y probar un modelo ajustado en nuevos datos.

## A continuación: Evaluación de clasificación de texto